<pre>
Project:        CT298DS004 Python Data Analytics Group Project
Programmer:     George Chang
Project Date:   2025-02-28
Filename:       fix_filename_space_dot_etc_plot_files_p.ipynb

In [7]:
### change characters in filesnames to underscore

import os
import numpy as np
import pandas as pd

path = '../plot_files_Peter/'

exclude_file_list = ['index.html']

ls_filename = []

### find all files
for root, dirs, files in os.walk(path):
    if root == path:
        for name in files:
            if name not in exclude_file_list:
                ls_filename.append(name)

sr_filename = pd.Series(ls_filename, name='filename')

df_all_page_files = pd.DataFrame(sr_filename)

def fix_one_html_file (row_idx : int):
    global path
    global df_all_page_files

    filename = df_all_page_files.iloc[row_idx].loc['filename']

    file_ext = '.unknown'
    file_ext_replace = '_unknown'

    if '.html' in filename:
        file_ext = '.html'
        file_ext_replace = '_html'
    elif '.png' in filename:
        file_ext = '.png'
        file_ext_replace = '_png'

    new_filename = filename.replace(' ', '_').replace('.', '_').replace('(', '_').replace(')', '_').replace('__', '_').replace(file_ext_replace, '') + file_ext

    if new_filename != filename:
        s_cmd = 'rename "' + path + filename + '" "' + path + new_filename + '"'
        os.system(s_cmd)


f1 = np.vectorize(fix_one_html_file, cache=True)(df_all_page_files.index)

# fix_one_html_file(0)


https://docs.python.org/3/library/os.html#os.system

In [37]:
### find files that only have png or html
### create dummy other file

import os
import numpy as np
import pandas as pd

path = '../plot_files_Peter/'

exclude_file_list = ['index.html']

ls_filename_html = []
ls_filename_png  = []

idx_html = []
idx_png  = []

### find all html and png files
for root, dirs, files in os.walk(path):
    if root == path:
        for name in files:
            if name not in exclude_file_list:
                if '.html' in name:
                    idx_name = name.replace('.html', '')
                    idx_html.append(idx_name)
                    ls_filename_html.append(name)
                elif '.png' in name:
                    idx_name = name.replace('.png', '')
                    idx_png.append(idx_name)
                    ls_filename_png.append(name)




sr_filename_html = pd.Series(ls_filename_html, index=idx_html, name='filename_html')
sr_filename_png  = pd.Series(ls_filename_png , index=idx_png , name='filename_png')

df_all_page_files = pd.DataFrame(sr_filename_html)
df_all_page_files = df_all_page_files.merge(sr_filename_png, how='outer', left_index=True, right_index=True)

def fix_one_file (old_filename_1, new_filename_1, new_filename_2):
    global path
    ### change old filename
    s_cmd = 'rename "' + path + old_filename_1 + '" "' + path + new_filename_1 + '"'
    os.system(s_cmd)
    ### create new file
    with open(path + new_filename_2, 'wt', encoding='utf-8') as f:
        f.write(' ')


def check_one_file (row_idx : int):
    global path
    global df_all_page_files

    filename_html = df_all_page_files.iloc[row_idx].loc['filename_html']
    filename_png  = df_all_page_files.iloc[row_idx].loc['filename_png']

    if pd.isna(filename_html):
        old_filename_1 = filename_png
        new_filename_1 = filename_png.replace('.png', '_no_html.png')
        new_filename_2 = filename_png.replace('.png', '_no_html.html')
        fix_one_file(old_filename_1, new_filename_1, new_filename_2)

    elif pd.isna(filename_png):
        old_filename_1 = filename_html
        new_filename_1 = filename_html.replace('.html', '_no_png.html')
        new_filename_2 = filename_html.replace('.html', '_no_png.png')
        fix_one_file(old_filename_1, new_filename_1, new_filename_2)



arr_idx_num = np.vectorize(df_all_page_files.index.get_loc, cache=True)(df_all_page_files.index.to_list())

f1 = np.vectorize(check_one_file, cache=True)(arr_idx_num)
